# AGN selection
@Author: Nyota Buduli Masirika

In [ ]:
#packages

import numpy as np
import pandas as pd
from astropy import units as u
import os 
from functions import *
from variables import *

In [ ]:
#Changement of folder (needed to work with ScopeSim)
os.chdir("ScopeSim") #relative path to ScopeSim folder
print(os.getcwd())

### i. Pc to mas converter and mas to pc converter
Function to convert the size in pc to the angular size in mas, using the distance to the source using:
$$\theta = 2 \arctan\left(\frac{S}{2D}\right)$$
with 
- $S$: the size in parsecs,
- $D$: the distance to the object in parsecs,
- $\theta$: the angular size in radians.

In [ ]:
# see functions.py

Function to convert the angular size in mas to the size in pc, using the distance to the source, as before.

In [ ]:
# see functions.py

## 1. Angular resolution of METIS 

To determine the angular resolution, we apply the Rayleigh criterion. It is expressed as follows:

$$\theta = 1.22 \frac{\lambda}{D}$$
with 
* $\theta$ the angular resolution,
* $\lambda$ the wavelength,
* $D$ the diameter.

In [ ]:
def Rayleigh_criterion(wavelength, diameter):
    """Apply the Rayleigh criterion to the telescope at the right wavelength

    Args:
        wavelength (Quantity): wavelength in meter
        diameter (Quantity): diameter of the mirror in meter

    Returns:
        Quantity: angular resolution of the telescope in mas
    """
    wavelength= wavelength.to(u.m)
    diameter = diameter.to(u.m)
    
    theta = 1.22*wavelength/diameter *u.rad
    theta = theta.to(u.mas)
    return theta

# Application of the Rayleigh criterion to ELT in the L-band and N-band
angular_resolution_L_band = Rayleigh_criterion(L_band, d_METIS) #[mas]
angular_resolution_N_band = Rayleigh_criterion(N_band, d_METIS) #[mas]
print(f"Angular resolutiuon in N-band =  {angular_resolution_N_band:.3f}")
print(f"Angular resolutiuon in L-band =  {angular_resolution_L_band:.3f}")

Angular resolutiuon in N-band =  76.838 mas
Angular resolutiuon in L-band =  25.613 mas


## 2. Angular size of 253 galaxies

To select the best candidates to make our observationswith METIS, we need to discriminate them with different criteria. The main one is the angular resolution of the AGN. To determine them, we use the database from the paper [*"The subarcsecond mid-infrared view of local active galactic nuclei: I. The N- and Q-band imaging atlas"* by D. Asmus et al.](https://arxiv.org/abs/1310.2770). The angular size of the galaxies are not directly communicated so we have to apply a bit of trigonometry. As we have the size of the source (in pc) and its distance, we can easily derive the angular size. 

Firts, we need to charge our database using the `pandas` library. The file is [database_galaxies.csv](./data/database_galaxies.csv).

In [ ]:
database_galaxies_csv = pd.read_csv("../data/database_galaxies.csv", usecols=["Object", "D", "F_nu(N-band)","d_unr"])
print(database_galaxies_csv)


              Object      D F_nu(N-band) d_unr
0                NaN  (Mpc)        (mJy)  (pc)
1        1H 0419-577  499.0           62   712
2     1RXS J112716.6  512.0           41   802
3    2MASX J03565655  351.0           31   541
4    2MASX J09180027  781.0           18   965
..               ...    ...          ...   ...
249      PKS 2354-35  222.0            2   346
250  Superantennae S  291.0          221   440
251         UGC 5101  182.0          227   597
252        UGC 12348  110.0           98   282
253          Z 41-20  170.0           29   238

[254 rows x 4 columns]


Then, we convert the size in pc to the angular size in mas, using the distance to the source. The angular size is then added to the database for future reference.
The formula is:
$$\theta = 2 \arctan\left(\frac{S}{2D}\right)$$
with 
- S: the diameter in parsecs,
- D: the distance to the object in parsecs,
- $\theta$: the angular diamter in radians.

(The function is written in [0. Functions and variables](#0-functions-and-variables)).

In [ ]:
database_galaxies_csv = pd.read_csv("../data/database_galaxies.csv", usecols=["Object", "D", "F_nu(N-band)","d_unr"], skiprows=[1]) #to remove the units from the database
database_galaxies = pd.DataFrame(database_galaxies_csv)
database_galaxies_np = pd.DataFrame(database_galaxies_csv).to_numpy()

#object_name = database_galaxies_np[:,0]
D = database_galaxies_np[:,1] * u.Mpc
S = database_galaxies_np[:,3] * u.pc
theta = np.array([(pc_to_mas(D[i], S[i])).value for i in range(len(D))]) *u.mas

database_galaxies["Angular size"] = theta.value.tolist()
# TODO: Add the units in the database
#database_galaxies.loc[0] = ["-", "[Mpc]", "[mJy]", "[pc]", "[mas]"]
print(database_galaxies)


              Object      D  F_nu(N-band)  d_unr  Angular size
0        1H 0419-577  499.0            62  712.0    294.309704
1     1RXS J112716.6  512.0            41  802.0    323.094482
2    2MASX J03565655  351.0            31  541.0    317.918120
3    2MASX J09180027  781.0            18  965.0    254.859844
4              3C 29  202.0             3  371.0    378.832887
..               ...    ...           ...    ...           ...
248      PKS 2354-35  222.0             2  346.0    321.475779
249  Superantennae S  291.0           221  440.0    311.878058
250         UGC 5101  182.0           227  597.0    676.593897
251        UGC 12348  110.0            98  282.0    528.787958
252          Z 41-20  170.0            29  238.0    288.770729

[253 rows x 5 columns]


Using the angular resolution limit of METIS, we can select the AGNs that we are able to resolve. We can see that they are all resolved. This database with the angular size is saved in [selected_galaxies.csv](./outputs/selected_galaxies.csv)

In [ ]:
selected_galaxies = database_galaxies[(database_galaxies['Angular size'] > angular_resolution_N_band)]
selected_galaxies.to_csv("../outputs/models/selected_galaxies.csv", encoding='utf-8', index=False)
print(selected_galaxies)

              Object      D  F_nu(N-band)  d_unr  Angular size
0        1H 0419-577  499.0            62  712.0    294.309704
1     1RXS J112716.6  512.0            41  802.0    323.094482
2    2MASX J03565655  351.0            31  541.0    317.918120
3    2MASX J09180027  781.0            18  965.0    254.859844
4              3C 29  202.0             3  371.0    378.832887
..               ...    ...           ...    ...           ...
248      PKS 2354-35  222.0             2  346.0    321.475779
249  Superantennae S  291.0           221  440.0    311.878058
250         UGC 5101  182.0           227  597.0    676.593897
251        UGC 12348  110.0            98  282.0    528.787958
252          Z 41-20  170.0            29  238.0    288.770729

[253 rows x 5 columns]
